# **タンパク質言語モデル入門**

## 概要
代表的なタンパク質言語モデル「ProtBERT」を使って、膜貫通型タンパク質を予測してみよう。

> **タンパク質言語モデル(ProtBERT)** とは
>  - 自然言語処理のために開発された技術（TransformerやBERT）を応用し、アミノ酸配列を「生命の言語」として学習させたタンパク質専用の大規模言語モデル
>  - 一般的な文章の代わりにタンパク質の配列データを読み込み、1024次元の数値ベクトルに変換
>  - 変換されたベクトルは、機能や構造、相互作用の予測、新たな機能タンパク質の探索などあらゆる用途に応用されている
>  - 原著論文：Elnaggar A, et al. (2022) ProtTrans: Toward Understanding the Language of Life Through Self-Supervised Learning, *IEEE Trans Pattern Anal Mach Intell*, 44(10):7112-7127

1. **課題** ：アミノ酸配列だけから、そのタンパク質が膜貫通型か可溶性かを識別する
   - **膜貫通型の例**: GPCR, チャネル, トランスポーター
   - **可溶性の例**: 酵素, 抗体, ホルモン

1. **埋め込みベクトルの可視化**
   - 次元削減とProtein Universeの可視化 — *類似したタンパク質は埋め込み空間で近くなるか*
   - 埋め込みベクトルの可視化 — *大規模言語モデルが、タンパク質をどのように認識しているかを直感的に理解する*

1. **機械学習による分類**
   - ロジスティック回帰
   - ランダムフォレスト
   - 両者の精度比較
   - 新たなタンパク質への適用

## 0. 必要ライブラリのインストールとインポート

In [ ]:
# 初回のみ実行してください
%pip install transformers torch scikit-learn pandas numpy matplotlib seaborn scipy requests matplotlib-fontja umap-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)
from sklearn.decomposition import PCA
import umap
import torch
from transformers import AutoTokenizer, AutoModel
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

import sys
# ---- 日本語フォントの設定(文字化け対策) ----
# 順序が重要:seaborn のテーマを「先に」設定すると rcParams を初期化するので、
# その「あと」で matplotlib-fontja を import すれば font.family が上書きされない。
print("kernel python :", sys.executable)
try:
    import matplotlib_fontja  # noqa: F401
    # 念のため font.sans-serif の先頭にも IPAexGothic を入れておく
    # (将来 sns.set_theme() などで font.family が 'sans-serif' に戻されても CJK が出るように)
    _sans = [f for f in plt.rcParams["font.sans-serif"] if f != "IPAexGothic"]
    plt.rcParams["font.sans-serif"] = ["IPAexGothic"] + _sans
    print("matplotlib-fontja: OK")
except ImportError as e:
    print(f"matplotlib-fontja import 失敗: {e}")
    print("→ 上のセル(%pip install ...)を先に実行してください。")
    from matplotlib import font_manager
    _candidates = [
        "IPAexGothic", "IPAGothic", "Noto Sans CJK JP", "Noto Sans JP",
        "TakaoGothic", "VL Gothic", "Hiragino Sans", "Yu Gothic", "Meiryo",
    ]
    _available = {f.name for f in font_manager.fontManager.ttflist}
    _picked = next((n for n in _candidates if n in _available), None)
    if _picked:
        plt.rcParams["font.family"] = _picked
        plt.rcParams["font.sans-serif"] = [_picked] + plt.rcParams["font.sans-serif"]
        print(f"フォールバックで {_picked!r} を使用します。")
    else:
        print("利用可能な CJK フォントが見つかりません。")
plt.rcParams["axes.unicode_minus"] = False

print("✓ ライブラリのインポート完了")

## 1. タンパク質言語モデルの準備

Hugging Face（AIモデルを共有するためのプラットフォーム）から事前学習済みのProtBERTモデルをダウンロードします。初回は数秒かかります。

### ProtBERTのダウンロード

In [ ]:
print("ProtBERT をロード中 (初回は1分程度かかります)...")

model_name = "Rostlab/prot_bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print(f"✓ ProtBERT ロード完了")
print(f"  デバイス: {device}")
print(f"  パラメータ数: {sum(p.numel() for p in model.parameters()):,}")

### アミノ酸配列をProtBERT埋め込みに変換する関数の定義

In [ ]:
def get_protein_embedding(sequence, tokenizer, model, device):
    
    sequence_spaced = " ".join(sequence)
    inputs = tokenizer(sequence_spaced, return_tensors="pt", max_length=4096,
                      truncation=True).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    
    return embedding

# テスト実行
test_sequence = "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVK"
test_embedding = get_protein_embedding(test_sequence, tokenizer, model, device)

print(f"✓ 埋め込み変換完了")
print(f"  入力シーケンス長: {len(test_sequence)} アミノ酸")
print(f"  出力ベクトル次元: {len(test_embedding)}")
print(f"  埋め込みベクトル(先頭10次元): {test_embedding[:10]}")

## 2. 訓練データの準備

### UniProtからヒトのタンパク質を取得
`N_PER_CLASS = 60`の数値を変えることで、ダウンロードするタンパク質の数を設定することができます。

In [ ]:
import requests
from io import StringIO

# 各クラスのタンパク質数(全体で 2 倍)。
# CPU で埋め込み計算する場合、60 で約 2-3 分、120 で約 5-6 分かかります。
# 試行錯誤の際は 30 程度に下げると軽快です。
N_PER_CLASS = 60

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")   # 標準 20 アミノ酸

# UniProt が取れない環境のためのフォールバック(各クラス 6 件)
FALLBACK_TM = {
    "beta2-adrenergic receptor": "MGQPGNGSAFLLAPNGSGLQWPHIIGQQPNGLSIGFQNGTSDVTAAYVQQVHNGVRWLF",
    "Mu opioid receptor": "MPPGNATAAGGPCPGLVLGVWVCSCVSIGVGYGQGHRSYSFSATRWQSAGVRSRCCCLA",
    "Voltage-gated potassium channel": "MSSGVAAGAGAGACGCMSCCCGALCCCCCATAGAGAGGPLSPTPACTDAPVPPPPTP",
    "Transferrin receptor": "MKSAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVK",
    "Glucose transporter 1": "MPPGPLPTHSQAQQGGQPQRARPPRPASAQCRAPPAAALPGPGHGSASDLPPPGAAP",
    "Rhodopsin": "MNGTEGPNFYVPFSNATGVVRSPFEYPQYYLAEPWQFSMLAAYMFLLIVLGFPINFLTLYVTVQHKKLR",
}
FALLBACK_SOL = {
    "Lysozyme": "MKALAVALATVFADYKDDDKGDPVQWWVNGKFLNGPYAKFDDNKRQGKYGFNDNEKR",
    "Hemoglobin (alpha)": "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGH",
    "Immunoglobulin G (Fab)": "DIQMTQSPSSLSASVGDRVTITCRASQSISSYLMHWVKQRPGKGLEWIGEIDPD",
    "Insulin": "GIVEQCCTSICSLYQLENYCN",
    "TNF-alpha": "MNLGHQSLGTLMALGLTALACGQDVKGWSVPAVPVAQSMYIFPAFSVDTCLVNVPVQL",
    "Carboxypeptidase A": "MLPKFLFLLAVSTINSHLALGDVPVQARQPGAALGMPPQMTTVPPFQGKDDELPSFK",
}


def fetch_uniprot(query, size):
    """UniProt REST API から TSV で SwissProt エントリを取得"""
    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query": query,
        "format": "tsv",
        "fields": "id,protein_name,sequence,length",
        "size": size,
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return pd.read_csv(StringIO(r.text), sep="\t")


def build_dataset():
    try:
        print(f"UniProt からヒトのレビュー済みタンパク質を取得中"
              f"(各クラス {N_PER_CLASS} 件、長さ 50–500)...")
        # 膜貫通型: SwissProt キーワード KW-0812 (Transmembrane)
        df_tm = fetch_uniprot(
            'keyword:KW-0812 AND reviewed:true AND organism_id:9606 '
            'AND length:[50 TO 500]',
            size=N_PER_CLASS,
        )
        # 可溶性: NOT Transmembrane
        df_sol = fetch_uniprot(
            'NOT keyword:KW-0812 AND reviewed:true AND organism_id:9606 '
            'AND length:[50 TO 500]',
            size=N_PER_CLASS,
        )
        df_tm["label"] = 1
        df_sol["label"] = 0
        df = pd.concat([df_tm, df_sol], ignore_index=True)
        df = df.rename(columns={"Entry Name": "name", "Sequence": "sequence"})
        df = df[["name", "sequence", "label"]].dropna(subset=["sequence"])
        # 非標準アミノ酸を含む配列を除外
        df = df[df["sequence"].apply(lambda s: set(s) <= VALID_AA)].reset_index(drop=True)
        print(f"✓ UniProt から {len(df)} タンパク質を取得")
        return df
    except Exception as e:
        print(f"⚠ UniProt 取得失敗 ({type(e).__name__}: {e})")
        print("→ 内蔵フォールバック(各クラス 6 件、計 12)を使用します")
        rows  = [{"name": n, "sequence": s, "label": 1} for n, s in FALLBACK_TM.items()]
        rows += [{"name": n, "sequence": s, "label": 0} for n, s in FALLBACK_SOL.items()]
        return pd.DataFrame(rows)


df_raw = build_dataset()

print()
print("データセット内訳:")
print(f"  膜貫通型: {(df_raw['label']==1).sum()}")
print(f"  可溶性  : {(df_raw['label']==0).sum()}")
print(f"  合計    : {len(df_raw)}")
print(f"\n最初の3行:")
print(df_raw.head(3))

## 3. 埋め込みベクトルの計算
時間がかかるのでのんびり待ちましょう。

In [ ]:
print(f"タンパク質埋め込みを計算中(全 {len(df_raw)} 件)...")
embeddings = []

step = max(1, len(df_raw) // 10)
for idx, row in df_raw.iterrows():
    seq = row["sequence"]
    try:
        embedding = get_protein_embedding(seq, tokenizer, model, device)
        embeddings.append(embedding)
        if (idx + 1) % step == 0 or (idx + 1) == len(df_raw):
            print(f"  {idx+1}/{len(df_raw)} 完了")
    except Exception as e:
        print(f"エラー (行 {idx}): {e}")
        embeddings.append(np.zeros(1280))

df_raw["embedding"] = embeddings
print(f"\n✓ 埋め込み計算完了: {len(embeddings)} タンパク質")

## 4. 埋め込みベクトルの可視化

In [ ]:
print("UMAP で次元削減中(4成分まで取得)...")
X_embeddings = np.array(df_raw["embedding"].tolist())

# n_neighbors は t-SNE の perplexity に相当(15 がデフォルト)。
# 少サンプルでは小さく、多サンプルでは 15-50 程度。
n_neighbors = min(30, max(5, len(X_embeddings) // 4))

# UMAP は 4 成分でも n_components パラメータだけで OK(t-SNE と違って barnes_hut 制約がない)
reducer = umap.UMAP(
    n_components=4,
    n_neighbors=n_neighbors,
    min_dist=0.1,
    random_state=42,
)
X_umap = reducer.fit_transform(X_embeddings)
print(f"✓ 次元削減完了 (n_neighbors={n_neighbors})")

for k in range(4):
    df_raw[f"umap_{k+1}"] = X_umap[:, k]

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
colors = {0: "red", 1: "white"}
labels = {0: "可溶性", 1: "膜貫通型"}

for ax, (a, b) in zip(axes.flat, [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]):
    for label in [0, 1]:
        mask = df_raw["label"] == label
        ax.scatter(
            df_raw.loc[mask, f"umap_{a}"],
            df_raw.loc[mask, f"umap_{b}"],
            c=colors[label],
            label=labels[label],
            s=60,
            alpha=0.7,
            edgecolors="black",
            linewidth=0.5,
        )
    ax.set_xlabel(f"UMAP 成分 {a}")
    ax.set_ylabel(f"UMAP 成分 {b}")
    ax.set_title(f"成分 {a} × 成分 {b}")
    ax.legend(fontsize=11)

fig.suptitle(
    f"ProtBERT 埋め込みのUMAP可視化（n={len(df_raw)}）",
    y=1.02, fontsize=13,
)
plt.tight_layout()
plt.show()

> **Q1：埋め込み空間において、可能性タンパク質と膜貫通型タンパク質は分離しているようにも混ざっているようにも見える。どちらなのか判断するためにできることはなんだろうか？**

## 5. 分類

### 訓練・テストデータの分割

In [ ]:
X = np.array(df_raw["embedding"].tolist())
y = df_raw["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"✓ データ分割完了")
print(f"  訓練データ: {len(X_train)} サンプル")
print(f"    - 膜貫通型: {(y_train == 1).sum()}")
print(f"    - 可溶性: {(y_train == 0).sum()}")
print(f"  テストデータ: {len(X_test)} サンプル")
print(f"    - 膜貫通型: {(y_test == 1).sum()}")
print(f"    - 可溶性: {(y_test == 0).sum()}")
print(f"\n  埋め込みベクトル次元: {X.shape[1]}")

### ロジスティック回帰による分類

In [ ]:
print("ロジスティック回帰を訓練中...")
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
y_pred_proba_lr = lr.predict_proba(X_test)[:, 1]

print(f"\n✓ ロジスティック回帰の結果")
print(f"  訓練精度: {lr.score(X_train, y_train):.3f}")
print(f"  テスト精度: {lr.score(X_test, y_test):.3f}")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_pred_proba_lr):.3f}")

print(f"\n分類レポート:")
print(classification_report(y_test, y_pred_lr,
                          target_names=["可溶性", "膜貫通型"]))

### ランダムフォレストによる分類

In [ ]:
print("ランダムフォレストを訓練中...")
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_pred_proba_rf = rf.predict_proba(X_test)[:, 1]

print(f"\n✓ ランダムフォレストの結果")
print(f"  訓練精度: {rf.score(X_train, y_train):.3f}")
print(f"  テスト精度: {rf.score(X_test, y_test):.3f}")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_pred_proba_rf):.3f}")

print(f"\n分類レポート:")
print(classification_report(y_test, y_pred_rf,
                          target_names=["可溶性", "膜貫通型"]))

### モデル比較

In [ ]:
comparison_df = pd.DataFrame({
    "モデル": ["ロジスティック回帰", "ランダムフォレスト"],
    "訓練精度": [
        lr.score(X_train, y_train),
        rf.score(X_train, y_train)
    ],
    "テスト精度": [
        lr.score(X_test, y_test),
        rf.score(X_test, y_test)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_pred_proba_lr),
        roc_auc_score(y_test, y_pred_proba_rf)
    ]
})

print("\n📊 モデル比較")
print(comparison_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

x = np.arange(len(comparison_df))
width = 0.25

axes[0].bar(x - width, comparison_df["訓練精度"], width, label="訓練精度", color="skyblue")
axes[0].bar(x, comparison_df["テスト精度"], width, label="テスト精度", color="coral")
axes[0].bar(x + width, comparison_df["ROC-AUC"], width, label="ROC-AUC", color="lightgreen")
axes[0].set_ylabel("スコア")
axes[0].set_title("モデル性能比較")
axes[0].set_xticks(x)
axes[0].set_xticklabels(comparison_df["モデル"])
axes[0].legend()
axes[0].set_ylim([0, 1.05])
axes[0].grid(axis="y", alpha=0.3)

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_pred_proba_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_pred_proba_rf)

axes[1].plot(fpr_lr, tpr_lr, label=f"LR (AUC={roc_auc_score(y_test, y_pred_proba_lr):.3f})", linewidth=2)
axes[1].plot(fpr_rf, tpr_rf, label=f"RF (AUC={roc_auc_score(y_test, y_pred_proba_rf):.3f})", linewidth=2)
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[1].set_xlabel("偽陽性率")
axes[1].set_ylabel("真陽性率")
axes[1].set_title("ROC 曲線")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 混同行列の可視化

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)
cm_rf = confusion_matrix(y_test, y_pred_rf)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(cm_lr, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["可溶性", "膜貫通型"],
            yticklabels=["可溶性", "膜貫通型"])
axes[0].set_title("ロジスティック回帰 - 混同行列")
axes[0].set_ylabel("実際のラベル")
axes[0].set_xlabel("予測ラベル")

sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Greens", ax=axes[1],
            xticklabels=["可溶性", "膜貫通型"],
            yticklabels=["可溶性", "膜貫通型"])
axes[1].set_title("ランダムフォレスト - 混同行列")
axes[1].set_ylabel("実際のラベル")
axes[1].set_xlabel("予測ラベル")

plt.tight_layout()
plt.show()

### ランダムフォレストの特徴量重要度

In [ ]:
feature_importance = pd.Series(
    rf.feature_importances_,
    index=[f"次元 {i}" for i in range(len(rf.feature_importances_))]
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
feature_importance.head(20).plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("特徴量重要度")
ax.set_title("ProtBERT 埋め込みの特徴量重要度(上位20)")
plt.tight_layout()
plt.show()

# === 最重要次元を取り出して次のセクション(Attention & 1D 可視化)で使う ===
top_dim = int(np.argmax(rf.feature_importances_))
print(f"\n→ 最大重要度の次元（top_dim）= 次元{top_dim}"
      f"(重要度 {rf.feature_importances_[top_dim]:.4f})")

> **Q2：シンプルな学習器でも、驚くほど良い精度で2種類のタンパク質を識別できた。その生物学的な理由を列挙してみよう**

## 6. 最重要埋め込み次元（`top_dim`）の可視化

上記のランダムフォレストで得られた **最重要次元 `top_dim`** を **線グラフ** （`hidden_state[..., top_dim]`の位置ごとの値）として描画してみます。

### `top_dim` を位置ごとに見る（**膜貫通型5個+可溶性5個**）

各位置の `hidden_state[..., top_dim]` を線グラフで描き、点を **アミノ酸の物理化学的性質**(疎水性 / 極性 / 正電荷 / 負電荷)で色分けします。

ランダムフォレストが最も重視した次元が、どの位置・どの性質のアミノ酸で強く反応しているかが直観的に分かります。

In [ ]:
# === 埋め込み次元 285 を位置ごとに可視化(5 TM + 5 可溶性、cell 16 のキャッシュを再利用)===

# --- アミノ酸の物理化学的分類 ---
AA_CATEGORY = {}
for aa in "AVLIMFWPG": AA_CATEGORY[aa] = "hydrophobic"   # 疎水性
for aa in "STNQYC":    AA_CATEGORY[aa] = "polar"         # 極性(無電荷)
for aa in "KRH":       AA_CATEGORY[aa] = "positive"      # 正電荷
for aa in "DE":        AA_CATEGORY[aa] = "negative"      # 負電荷

CATEGORY_COLORS = {
    "hydrophobic": "#F4A261",   # オレンジ
    "polar":       "#2A9D8F",   # ティール
    "positive":    "#264653",   # 濃い青(K, R, H)
    "negative":    "#E76F51",   # 赤(D, E)
}
CATEGORY_LABELS = {
    "hydrophobic": "疎水性 (A,V,L,I,M,F,W,P,G)",
    "polar":       "極性無電荷 (S,T,N,Q,Y,C)",
    "positive":    "正電荷 (K,R,H)",
    "negative":    "負電荷 (D,E)",
}

# --- 2 × 5 グリッド ---
fig, axes = plt.subplots(2, 5, figsize=(22, 8))
for ax, r in zip(axes.flat, protein_outputs):
    seq, vals = r["seq"], r["hidden_dim"]
    colors = [CATEGORY_COLORS[AA_CATEGORY.get(aa, "polar")] for aa in seq]
    ax.plot(range(len(seq)), vals, color="gray", linewidth=0.8, alpha=0.4, zorder=1)
    ax.scatter(range(len(seq)), vals, c=colors, s=30,
               edgecolors="black", linewidth=0.3, zorder=3)
    ax.axhline(0, color="black", linewidth=0.4)
    cls = "膜貫通" if r["label"] == 1 else "可溶性"
    ax.set_title(f"[{cls}] {r['name']}\n({len(seq)} aa)", fontsize=9)
    if len(seq) <= 60:
        ax.set_xticks(range(len(seq)))
        ax.set_xticklabels(list(seq), fontsize=5)
    else:
        ax.tick_params(axis="x", labelsize=6)
    ax.tick_params(axis="y", labelsize=7)

# 凡例(共通)
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", label=CATEGORY_LABELS[cat],
           markerfacecolor=color, markeredgecolor="black", markersize=9)
    for cat, color in CATEGORY_COLORS.items()
]
fig.legend(handles=legend_elements, loc="lower center",
           bbox_to_anchor=(0.5, -0.02), ncol=4, fontsize=10)
fig.suptitle(
    f"hidden_state[..., {hidden_dim}]: 上段 = 膜貫通型、下段 = 可溶性\n"
    "色 = アミノ酸の物理化学的性質",
    y=1.02, fontsize=13,
)
plt.tight_layout()
plt.show()

### Attention重みの可視化
ついでに、ProtBERTの中身をイメージするために、Attention重み（とある（layer, head）の重み行列）もAttention ヒートマップとして可視化しましょう。

In [ ]:
# === ProtBERT の attention を 5 TM + 5 可溶性タンパク質で可視化 ===
import torch

# 第 12 章 RF で得た top_dim を使う(attention/hidden は別空間だが教育上同じ番号を採用)
n_attn_total = 30 * 16          # ProtBERT: 30 層 × 16 ヘッド = 480 個
flat_idx   = top_dim % n_attn_total   # 0..479 にラップ
hidden_dim = top_dim                  # 0..1023 そのまま
print(f"top_dim={top_dim} → flat_idx={flat_idx}, hidden_dim={hidden_dim}")

# 5 TM + 5 可溶性(各クラスで最短の 5 タンパク質)
_targets = pd.concat([
    df_raw[df_raw["label"] == 1]
        .assign(_len=lambda d: d["sequence"].str.len()).nsmallest(5, "_len"),
    df_raw[df_raw["label"] == 0]
        .assign(_len=lambda d: d["sequence"].str.len()).nsmallest(5, "_len"),
])
print(f"対象: {len(_targets)} タンパク質(膜貫通 5 + 可溶性 5)")

# --- 1 回の推論で attention と hidden state の必要部分だけキャッシュ ---
protein_outputs = []      # ← cell 18 でも再利用
n_heads = None
for _, row in _targets.iterrows():
    seq = row["sequence"]
    spaced = " ".join(list(seq))
    inputs = tokenizer(spaced, return_tensors="pt",
                       truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out = model(**inputs, output_attentions=True)
    if n_heads is None:
        n_heads = out.attentions[0].shape[1]
        layer_idx, head_idx = divmod(flat_idx, n_heads)
        print(f"flat index {flat_idx} → layer={layer_idx}, head={head_idx} "
              f"(層数 {len(out.attentions)}, ヘッド数 {n_heads})")
    protein_outputs.append({
        "name":  row["name"],
        "seq":   seq,
        "label": int(row["label"]),
        "attention":   out.attentions[layer_idx][0, head_idx].cpu().numpy()[1:-1, 1:-1],
        "hidden_dim":  out.last_hidden_state[0, 1:-1, hidden_dim].cpu().numpy(),
    })

# --- 2 × 5 グリッドで attention ヒートマップ ---
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
for ax, r in zip(axes.flat, protein_outputs):
    im = ax.imshow(r["attention"], cmap="viridis", aspect="auto")
    cls = "膜貫通" if r["label"] == 1 else "可溶性"
    ax.set_title(f"[{cls}] {r['name']}\n({len(r['seq'])} aa)", fontsize=9)
    if len(r["seq"]) <= 60:
        ax.set_xticks(range(len(r["seq"])))
        ax.set_yticks(range(len(r["seq"])))
        ax.set_xticklabels(list(r["seq"]), fontsize=5)
        ax.set_yticklabels(list(r["seq"]), fontsize=5)
    else:
        step = max(1, len(r["seq"]) // 10)
        ax.set_xticks(range(0, len(r["seq"]), step))
        ax.set_yticks(range(0, len(r["seq"]), step))
        ax.tick_params(axis="both", labelsize=6)

fig.suptitle(
    f"ProtBERT attention (layer {layer_idx}, head {head_idx} / flat index {flat_idx})\n"
    "上段 = 膜貫通型(label=1)、下段 = 可溶性(label=0)",
    y=1.02, fontsize=13,
)
fig.colorbar(im, ax=axes, fraction=0.015, pad=0.02, label="attention weight")
plt.show()

> **Q3：線グラフを見ながら、なぜこの`top_dim`が識別に大きく寄与できたのか考えてみよう**

## 7. 新しいタンパク質での予測

In [ ]:
new_proteins = {
    "Aquaporin-1 (膜蛋白)": "MSKGEQVSTAQESVQEAKS",
    "GFP (可溶性)": "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVWPT",
}

print("\n🔍 新しいタンパク質での予測\n")

for protein_name, sequence in new_proteins.items():
    embedding = get_protein_embedding(sequence, tokenizer, model, device)
    
    pred_lr = lr.predict([embedding])[0]
    prob_lr = lr.predict_proba([embedding])[0]
    
    pred_rf = rf.predict([embedding])[0]
    prob_rf = rf.predict_proba([embedding])[0]
    
    pred_label_lr = "膜貫通型" if pred_lr == 1 else "可溶性"
    pred_label_rf = "膜貫通型" if pred_rf == 1 else "可溶性"
    
    print(f"タンパク質: {protein_name}")
    print(f"  配列長: {len(sequence)} アミノ酸")
    print(f"  ロジスティック回帰: {pred_label_lr} (確信度: {max(prob_lr):.1%})")
    print(f"  ランダムフォレスト: {pred_label_rf} (確信度: {max(prob_rf):.1%})")
    print()

## 8. じっくり考えよう

1. **埋め込みベクトルの意味**
   - ロジスティック回帰やランダムフォレストといったシンプルな分類機で十分高い予測精度が出たが、その理由はなんだろう？

1. **埋め込みベクトルの解釈**
   - どのようにすれば、ProtBERTの埋め込みベクトルを解釈できるだろうか？

1. **タンパク質言語モデルの将来**
   - 日々、タンパク質言語モデルの新しい活用法が編み出されている。この１年で発表された斬新な論文を１本見つけて、建設的に批判してみよう